# 32 — CycleGAN-Turbo baseline (revision round 1, reviewer R2-4)

Same 298 crops, masks and export as MRT-GAN. 30 steps, train size ladder. Train crops only; the test split is not read.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/turbo_baseline_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# Study (p100 crops and configs) from kernel 13 output; three G3 tuning configs written next to G0-G3.
if torch.cuda.device_count()<1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count())
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
P100=STUDY/'p100'
sys.path.insert(0,str(HERE))
from melon_cyclegan import texture_study as s
from melon_cyclegan import texture_study_eval as e
scope,m=s.verify_scope(STUDY,100,DATASET)
g0=json.loads((P100/'configs/G0.json').read_text()); g3=json.loads((P100/'configs/G3.json').read_text())
assert g0['subset_digest']==g3['subset_digest']==m['subset_digest']
# R512 pool: same crop windows, same reviewed-pool membership and same GrabCut masks (nearest-upsampled),
# images re-cropped from the original photographs at 512 px. Only resolution changes.
def build_pool512(src, dst, dataset, size=512, require_train=True):
    import csv, json, collections, hashlib
    from pathlib import Path
    import numpy as np, cv2
    from PIL import Image
    from melon_cyclegan import config as cfg
    from melon_cyclegan.crops import crop_with_reflection
    from melon_cyclegan.color import phi_numpy
    from melon_cyclegan.training_data import digest, verify, sha
    src, dst, dataset = Path(src), Path(dst), Path(dataset)
    if dst.exists(): raise ValueError(f'{dst} exists')
    man = json.loads((src/'crops_manifest.json').read_text())
    rows = {r['id']: r for r in csv.DictReader(open(dataset/'manifest.csv'))}
    k = size/float(man.get('size', 256)); records=[]; feats=collections.defaultdict(list); drift=[]
    for r in man['records']:
        row = rows[r['id']]
        if (require_train and row['split']!='train') or sha(dataset/row['image'])!=r['source_sha256']: raise ValueError(f"{r['id']}: source photo differs")
        with Image.open(dataset/row['image']) as h: img=h.convert('RGB')
        left, top, side = r['crop_window']
        crop = cv2.resize(crop_with_reflection(img,left,top,side),(size,size),interpolation=cv2.INTER_AREA)
        m256 = Image.open(src/r['mask']).convert('L')
        mask = np.asarray(m256.resize((size,size),Image.Resampling.NEAREST))
        old = np.asarray(Image.open(src/r['image']).convert('RGB'),np.float64)
        drift.append(np.abs(cv2.resize(crop,old.shape[1::-1],interpolation=cv2.INTER_AREA).astype(np.float64)-old).mean())
        rec = dict(r, crop_box=[float(v)*k for v in r['crop_box']])
        for key, arr in (('image', crop), ('mask', mask)):
            p = dst/r[key]; p.parent.mkdir(parents=True, exist_ok=True); Image.fromarray(arr).save(p); rec[f'{key}_sha256']=sha(p)
        feats[int(r['class_id'])].append(phi_numpy(crop.astype(np.float64)/255.0, mask>127)); records.append(rec)
    if max(drift) > 4.0: raise ValueError(f'512 crops do not match the 256 pool geometry (max mean abs diff {max(drift):.2f})')
    classes = {}
    for cid, v in sorted(feats.items()):
        x=np.stack(v); classes[cfg.CLASS_NAMES[cid]]=dict(class_id=cid,count=len(x),centroid=x.mean(0).tolist(),spread=x.std(0).tolist())
    new = {k2: v for k2, v in man.items() if k2 not in ('records','crops_digest','centroids_sha256')}
    new.update(records=records, size=size, crops_digest=digest(records), source_crops_digest_256=man['crops_digest'],
               note='R512: same windows, members and GrabCut masks as the 256 pool; images re-cropped at 512')
    (dst/'color_centroids.json').write_text(json.dumps(dict(version=cfg.VERSION, phi=man['phi'],
        source='curated train split real crops (R512)', crops_digest=new['crops_digest'], classes=classes), indent=2)+'\n')
    new['centroids_sha256'] = sha(dst/'color_centroids.json')
    (dst/'crops_manifest.json').write_text(json.dumps(new, indent=2)+'\n')
    verify(dst, man['training_policy'])
    return new, dict(max_drift=float(max(drift)), mean_drift=float(np.mean(drift)), n=len(records))

POOL512=P100/'crops512'
pool_manifest,pool_check=build_pool512(P100/'crops',POOL512,DATASET)
print('R512 pool',pool_check,pool_manifest['crops_digest'])
sys.path.insert(0,str(HERE/'tools'))
from calibrate_color_margin import calibrate
report=calibrate(POOL512); (POOL512/'color_margin.json').write_text(json.dumps(report,indent=2)+'\n')
print('margin 512',report['margin'],'margin 256',json.loads((P100/'crops/color_margin.json').read_text())['margin'])
if report['margin']<=0: raise ValueError('nonpositive margin at 512')
from melon_cyclegan.dataset_impl import load_pair
from melon_cyclegan.texture import patch_indices
for r in pool_manifest['records']:
    _,mk=load_pair(POOL512/r['image'],POOL512/r['mask']); patch_indices(mk[None],64,32,4,.95,random=False)
g3ti=dict(g3,composite_mask=0,lambda_identity=0.5,load_size=512,crop_size=512,texture_size=64,texture_stride=32,
          expected_crops_digest=pool_manifest['crops_digest'])


In [ ]:
# Isolated Python 3.10 environment with the versions of the repository's environment.yaml; repository pinned by commit.
TURBO=Path('/tmp/turbo'); PY='/tmp/tenv/bin/python'
sh=lambda cmd: subprocess.run(cmd,shell=True,check=True)
sh('pip install -q uv && uv venv --python 3.10 /tmp/tenv')
sh(f'uv pip install -q --python {PY} torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121')
sh(f'uv pip install -q --python {PY} diffusers==0.25.1 transformers==4.35.2 huggingface_hub==0.20.3 accelerate==0.25.0 peft==0.7.1 "numpy<2" lpips clean-fid vision_aided_loss wandb einops tqdm opencv-python-headless scipy timm "setuptools<70" ftfy regex dominate git+https://github.com/openai/CLIP.git')
sh(f'git clone -q https://github.com/GaParmar/img2img-turbo.git {TURBO} && git -C {TURBO} checkout -q 86f54146590ffb4543c8cf85b5a36657da670924')
# Author training prep (resize 1.117x, random crop, flip) at a chosen size; the only change to the repository.
tu=TURBO/'src/my_utils/training_utils.py'; t_=tu.read_text()
old='    elif image_prep in ["resize_256", "resize_256x256"]:'
new="""    elif image_prep.startswith('scaled_'):
        S = int(image_prep.split('_')[1])
        T = transforms.Compose([
            transforms.Resize((round(S*286/256), round(S*286/256)), interpolation=Image.LANCZOS),
            transforms.RandomCrop((S, S)),
            transforms.RandomHorizontalFlip(),
        ])
"""+old
assert t_.count(old)==1; tu.write_text(t_.replace(old,new))


In [ ]:
# Same 298 pool crops as MRT-GAN: train_A ripe (217), train_B unripe (81). test_A/B hold four training crops because the script needs them; no validation or test image is read.
import random
DS=WORK/'turbo_data'
for d in ('train_A','train_B','test_A','test_B'): (DS/d).mkdir(parents=True)
recs=pool_manifest['records']; assert all(r['split']=='train' for r in recs)
for r in recs:
    shutil.copy(POOL512/r['image'],DS/('train_B' if r['domain']=='A' else 'train_A')/f"{r['id']}.png")
for d in 'AB':
    for f in sorted((DS/f'train_{d}').glob('*.png'))[:4]: shutil.copy(f,DS/f'test_{d}'/f.name)
assert len(list((DS/'train_A').iterdir()))==217 and len(list((DS/'train_B').iterdir()))==81
(DS/'fixed_prompt_a.txt').write_text('a photo of a ripe melon')
(DS/'fixed_prompt_b.txt').write_text('a photo of an unripe melon')


In [ ]:
# Author loss weights and learning rate. One process per GPU. Size: largest of 448, 384, 320, 256 that trains on a T4 with gradient checkpointing.
import threading, time, re
STEPS=30; N_EXPORT=6; RES=0; SEEDS=[11,12,13] if RES else [11]
ENV=dict(os.environ,WANDB_MODE='disabled',PYTHONPATH=str(TURBO/'src'),HF_HUB_DISABLE_TELEMETRY='1',PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',ACCELERATE_MIXED_PRECISION='fp16')
peak=[0,0]
def sample():
    while not done.is_set():
        q=subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits',shell=True,capture_output=True,text=True).stdout.split()
        for i,v in enumerate(q[:2]): peak[i]=max(peak[i],int(v))
        time.sleep(5)
def cmd(seed,res,steps):
    return [PY,str(TURBO/'src/train_cyclegan_turbo.py'),'--pretrained_model_name_or_path','stabilityai/sd-turbo','--output_dir',str(WORK/f'turbo_s{seed}'),
      '--dataset_folder',str(DS),'--train_img_prep',f'scaled_{res}','--val_img_prep','resize_512','--learning_rate','1e-5','--max_train_steps',str(steps),
      '--train_batch_size','1','--gradient_accumulation_steps','1','--report_to','wandb','--tracker_project_name','melon','--validation_steps',str(steps),
      '--checkpointing_steps',str(steps-1),'--viz_freq',str(steps),'--lambda_gan','0.5','--lambda_idt','1','--lambda_cycle','1','--seed',str(seed),'--gradient_checkpointing']
def run(seed,gpu,res,steps):
    log=open(WORK/f'turbo_s{seed}.log','w'); t=time.time()
    r=subprocess.run(cmd(seed,res,steps),env=dict(ENV,CUDA_VISIBLE_DEVICES=str(gpu)),stdout=log,stderr=subprocess.STDOUT,cwd=TURBO)
    txt=open(WORK/f'turbo_s{seed}.log',errors='ignore').read().replace('\r','\n')
    rate=re.findall(r'Steps:\s+\d+%.*?(\d+)/\d+ \[[^\]]*?([\d.]+(?:it/s|s/it))',txt)
    print('seed',seed,'gpu',gpu,'res',res,'rc',r.returncode,'minutes',round((time.time()-t)/60,1),'last rate',rate[-1] if rate else None,flush=True)
    if r.returncode: print(txt[-2200:],flush=True)
    return r.returncode
done=threading.Event(); threading.Thread(target=sample,daemon=True).start()
results={}
if RES:
    plan={0:[11,13],1:[12]}
    def worker(g):
        for sd in plan[g]: results[sd]=run(sd,g,RES,STEPS)
    th=[threading.Thread(target=worker,args=(g,)) for g in plan]; [t.start() for t in th]; [t.join() for t in th]
else:
    for res in (448,384,320,256):
        shutil.rmtree(WORK/'turbo_s11',ignore_errors=True)
        results[11]=run(11,0,res,STEPS)
        if results[11]==0: RES=res; break
done.set(); print('chosen res',RES,'peak MiB',peak,results,flush=True)
if any(results.values()): raise SystemExit('training failed')


In [ ]:
# Export ripe-to-unripe on EMA-free one-step model at 512 px, then paste through the same masks and code as the other arms.
from PIL import Image
import csv
(WORK/'turbo_infer.py').write_text("""
import sys, json, torch
from pathlib import Path
from PIL import Image
from torchvision import transforms
sys.path.insert(0, sys.argv[1]+'/src')
from cyclegan_turbo import CycleGAN_Turbo
ck, src, dst, n = sys.argv[2], Path(sys.argv[3]), Path(sys.argv[4]), int(sys.argv[5])
m = CycleGAN_Turbo(pretrained_path=ck); m.eval(); dst.mkdir(parents=True, exist_ok=True)
for f in sorted(src.glob('*.png'))[:n]:
    x = transforms.Normalize([0.5],[0.5])(transforms.ToTensor()(Image.open(f).convert('RGB'))).unsqueeze(0).cuda()
    with torch.no_grad(): y = m(x, direction='a2b', caption='a photo of an unripe melon')
    transforms.ToPILImage()((y[0].cpu()*0.5+0.5).clamp(0,1)).save(dst/f.name)
""")
from melon_cyclegan import config as cfg
gbc=s.tool('generate_balance_candidates')
rows={r['id']:r for r in csv.DictReader(open(DATASET/'manifest.csv'))}
byid={r['id']:r for r in pool_manifest['records']}
OUT=WORK/'result'; OUT.mkdir()
for sd in SEEDS:
    ck=WORK/f'turbo_s{sd}/checkpoints/model_{STEPS}.pkl'
    raw=WORK/f'raw_s{sd}'
    subprocess.run([PY,str(WORK/'turbo_infer.py'),str(TURBO),str(ck),str(DS/'train_A'),str(raw),str(N_EXPORT)],env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),check=True)
    d=OUT/f'candidates/TURBO_s{sd}'
    for sub in ('full/images','full/labels','images'): (d/sub).mkdir(parents=True)
    recs_=[]
    for f in sorted(raw.glob('*.png')):
        crop=byid[f.stem]; row=rows[f.stem]
        with Image.open(DATASET/row['image']) as h: full=h.convert('RGB')
        syn,clip=gbc.paste(full,Image.open(f).convert('RGB'),Image.open(POOL512/crop['mask']).convert('L'),crop['crop_window'])
        cid=f'balance__{f.stem}'
        syn.save(d/'full/images'/f'{cid}.jpg',quality=95)
        (d/'full/labels'/f'{cid}.txt').write_text(gbc.flipped_label((DATASET/row['label']).read_text(),cfg.UNRIPE))
        l,t,sz=crop['crop_window']; syn.crop((l,t,l+sz,t+sz)).resize((512,512),Image.Resampling.LANCZOS).save(d/'images'/f'{cid}.png')
        recs_.append(dict(candidate_id=cid,parent_id=f.stem,full_image=f'full/images/{cid}.jpg',full_label=f'full/labels/{cid}.txt',image=f'images/{cid}.png'))
    (d/'records.json').write_text(json.dumps(recs_,indent=1))
    (OUT/f'checkpoints').mkdir(exist_ok=True); shutil.copy(ck,OUT/f'checkpoints/turbo_s{sd}.pkl'); shutil.copy(WORK/f'turbo_s{sd}.log',OUT/f'train_s{sd}.log')
    print('seed',sd,'candidates',len(recs_),flush=True)
(OUT/'summary.json').write_text(json.dumps(dict(baseline='CycleGAN-Turbo',commit='86f54146590ffb4543c8cf85b5a36657da670924',steps=STEPS,train_size=RES,seeds=SEEDS,peak_mib=peak,test_used=False),indent=1))
for name in ('turbo_data','raw_s11','raw_s12','raw_s13','turbo_s11','turbo_s12','turbo_s13','study','code'): shutil.rmtree(WORK/name,ignore_errors=True)
print('done',sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file())//2**20,'MiB')
